# Notebook 00 — Dataset Cleaning

Produces silence-trimmed copies of the original audio files in `audio_cleaned/`,
which mirrors the structure of `audio/` but with leading/trailing silence removed
and clips below a minimum duration rejected.

**Pipeline**
1. Load each original `.wav` and resample to 16 kHz
2. Trim leading/trailing silence via frame-level RMS energy gate
3. Reject clips shorter than `MIN_DURATION` seconds after trimming
4. Save trimmed clips to `audio_cleaned/{label}/`
5. Write `cleaning_summary.csv` and `cleaning_rejected.csv` manifests

**Output consumed by**: `01-prepare-augmented-data_current.ipynb`

In [ ]:
import os
import re
import numpy as np
import torch
import torchaudio
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path
from tqdm.notebook import tqdm

In [ ]:
def extract_speaker(filename: str) -> str:
    """
    Extract and normalise a speaker name from an audio filename.

    Strips all trailing _N clip/segment suffixes and lowercases the result.
    Handles single-suffix (name_3) and double-suffix (name_5_1) patterns
    that appear in this dataset.

    Examples:
        daningram_15        → daningram
        aileenhernandez_5_1 → aileenhernandez   (double suffix — handled correctly)
        johnmackey_15_1     → johnmackey         (double suffix — handled correctly)
        JimmyCalderwood_5   → jimmycalderwood    (case normalised)
    """
    name = re.sub(r'_aug\d+$', '', filename)  # strip augmentation suffix first
    name = re.sub(r'(_\d+)+$', '', name)       # strip all trailing _N groups in one pass
    return name.strip().lower()

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Configuration

In [ ]:
save_path   = Path('/content/drive/MyDrive/CS7357/Project/data')
raw_audio   = save_path / 'audio'          # original recordings
clean_audio = save_path / 'audio_cleaned'  # output — same structure, silence trimmed

TARGET_SR    = 16_000  # resample everything to 16 kHz (matches wav2vec2 expectation)
TOP_DB       = 40      # dB below peak to treat as silence
MIN_DURATION = 5.0     # seconds — reject clips shorter than this after trimming
LABELS       = ['dementia', 'nodementia']

print(f'Source      : {raw_audio}')
print(f'Destination : {clean_audio}')
print(f'Target SR   : {TARGET_SR} Hz')
print(f'Silence gate: {TOP_DB} dB below peak')
print(f'Min duration: {MIN_DURATION} s after trim')

## Silence Trimming Function

Uses a frame-level RMS energy gate: any frame whose RMS is more than `TOP_DB`
decibels below the clip peak is considered silent. Leading and trailing silent
frames are removed; interior silence is preserved.

In [ ]:
def trim_silence(wav: torch.Tensor, top_db: float = 40, frame_len: int = 512, hop_len: int = 256) -> torch.Tensor:
    """
    Remove leading and trailing silence from a waveform tensor.

    Args:
        wav      : (C, T) or (T,) waveform tensor — converted to mono internally
        top_db   : dB below peak RMS to treat as silent
        frame_len: analysis frame size in samples
        hop_len  : hop between frames in samples

    Returns:
        (1, T') trimmed mono waveform tensor, or the original if fully silent
    """
    # collapse to mono
    if wav.dim() == 1:
        wav = wav.unsqueeze(0)
    if wav.shape[0] > 1:
        wav = wav.mean(dim=0, keepdim=True)

    wav_np = wav.squeeze().numpy()

    # frame-level RMS
    n_frames = max(1, (len(wav_np) - frame_len) // hop_len)
    rms = np.array([
        np.sqrt(np.mean(wav_np[i * hop_len : i * hop_len + frame_len] ** 2))
        for i in range(n_frames)
    ])

    peak_rms  = rms.max()
    if peak_rms == 0:
        return wav  # fully silent — caller will reject

    threshold = peak_rms / (10 ** (top_db / 20.0))
    active    = np.where(rms >= threshold)[0]

    if len(active) == 0:
        return wav  # fully silent — caller will reject

    start = int(active[0]  * hop_len)
    end   = int(active[-1] * hop_len + frame_len)
    end   = min(end, wav.shape[-1])

    return wav[:, start:end]

## Cleaning Loop

In [ ]:
records  = []  # kept files — summary stats
rejected = []  # rejected files — reason

for label in LABELS:
    src_dir = raw_audio   / label
    dst_dir = clean_audio / label
    dst_dir.mkdir(parents=True, exist_ok=True)

    wav_files = sorted(src_dir.rglob('*.wav'))
    print(f'\n{label}: {len(wav_files)} source files')

    for wav_path in tqdm(wav_files, desc=label):
        try:
            wav, sr = torchaudio.load(wav_path)

            # resample to target SR if needed
            if sr != TARGET_SR:
                wav = torchaudio.transforms.Resample(sr, TARGET_SR)(wav)

            original_dur = wav.shape[-1] / TARGET_SR

            trimmed     = trim_silence(wav, top_db=TOP_DB)
            trimmed_dur = trimmed.shape[-1] / TARGET_SR

            # reject clips that are too short after trimming
            if trimmed_dur < MIN_DURATION:
                rejected.append({
                    'file'  : wav_path.name,
                    'label' : label,
                    'reason': f'only {trimmed_dur:.1f}s of audio after silence trim (min {MIN_DURATION}s)',
                })
                continue

            dst_path = dst_dir / wav_path.name
            torchaudio.save(str(dst_path), trimmed, TARGET_SR)

            records.append({
                'file'       : wav_path.stem,
                'label'      : label,
                'speaker'    : extract_speaker(wav_path.stem),
                'original_s' : round(original_dur, 2),
                'trimmed_s'  : round(trimmed_dur,  2),
                'removed_s'  : round(original_dur - trimmed_dur, 2),
            })

        except Exception as e:
            rejected.append({'file': wav_path.name, 'label': label, 'reason': str(e)})

summary_df  = pd.DataFrame(records)
rejected_df = pd.DataFrame(rejected)

print(f'\n=== Cleaning Summary ===')
print(f'Kept    : {len(summary_df)}')
print(f'Rejected: {len(rejected_df)}')
if not summary_df.empty:
    print()
    print(summary_df.groupby('label')[['original_s', 'trimmed_s', 'removed_s']].mean().round(2))

## Visualise Trim Effect

In [ ]:
fig, axes = plt.subplots(1, len(LABELS), figsize=(6 * len(LABELS), 4))

for ax, label in zip(axes, LABELS):
    sub = summary_df[summary_df['label'] == label]
    ax.hist(sub['original_s'], bins=20, alpha=0.6, label='Original')
    ax.hist(sub['trimmed_s'],  bins=20, alpha=0.6, label='Trimmed')
    ax.set_title(f'{label} — duration distribution')
    ax.set_xlabel('Duration (s)')
    ax.set_ylabel('Count')
    ax.legend()

plt.suptitle('Audio duration before and after silence trimming', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
print('=== Dataset Integrity Checks ===\n')

# ── 1. Both classes present in cleaned output ────────────────────────────────
print('1. Class coverage:')
all_ok = True
for label in LABELS:
    count = len(summary_df[summary_df['label'] == label])
    status = '✓' if count > 0 else '✗ MISSING'
    print(f'   {status}  {label}: {count} files, '
          f"{summary_df[summary_df['label'] == label]['speaker'].nunique()} unique speakers")
    if count == 0:
        all_ok = False
print()

# ── 2. Speaker name normalisation check ──────────────────────────────────────
print('2. Speaker name normalisation:')
suspect = summary_df[summary_df['speaker'].str.match(r'.*_\d+$')]
if suspect.empty:
    print('   ✓  All speaker names normalised correctly (no trailing _N digits)')
else:
    print(f'   ⚠  {len(suspect)} files have suspect speaker names:')
    print(suspect[['file', 'speaker']].to_string(index=False))
print()

# ── 3. No speaker appears in both classes ────────────────────────────────────
print('3. Cross-class speaker integrity:')
dm_speakers = set(summary_df[summary_df['label'] == 'dementia']['speaker'])
nd_speakers = set(summary_df[summary_df['label'] == 'nodementia']['speaker'])
cross = dm_speakers & nd_speakers
if cross:
    print(f'   ⚠  {len(cross)} speaker(s) appear in BOTH classes — possible label error:')
    for s in sorted(cross):
        print(f'      {s}')
else:
    print('   ✓  No speaker appears in both dementia and nodementia')
print()

# ── 4. Summary table ─────────────────────────────────────────────────────────
print('4. Per-class summary:')
print(summary_df.groupby('label').agg(
    files=('file', 'count'),
    speakers=('speaker', 'nunique'),
    avg_duration_s=('trimmed_s', 'mean'),
    min_duration_s=('trimmed_s', 'min'),
    max_duration_s=('trimmed_s', 'max'),
).round(2).to_string())

## Save Manifests

In [ ]:
summary_df[['file', 'label', 'speaker', 'original_s', 'trimmed_s', 'removed_s']].to_csv(
    save_path / 'cleaning_summary.csv', index=False
)
print(f'Saved cleaning_summary.csv  ({len(summary_df)} rows, columns: file, label, speaker, original_s, trimmed_s, removed_s)')

if not rejected_df.empty:
    rejected_df.to_csv(save_path / 'cleaning_rejected.csv', index=False)
    print(f'Saved cleaning_rejected.csv ({len(rejected_df)} rows)')
    print()
    print('Rejected files:')
    print(rejected_df.to_string(index=False))

print()
print('Next step: run 01-prepare-augmented-data_current.ipynb')
print(f'  dm_path = {clean_audio / "dementia"}')
print(f'  nd_path = {clean_audio / "nodementia"}')